> **Production run.** Same notebook with the numbers of replicas, toy experiments and chain lengths of the report. Run with `run_full.sh`; expect several hours.

# 01 · Frequentist analysis (1/4): data, model and fit of $A_{\rm raw}$ in $B^\pm\to K^\pm K^+K^-$

**What this notebook does.** From the raw candidates of the LHCb Open Data to the value of $A_{\rm raw}=[N(B^-)-N(B^+)]/[N(B^-)+N(B^+)]$
with its uncertainty and significance: preselection, mass model, extended unbinned likelihood fitted with `iminuit`,
Minos intervals, profile likelihood and Wilks test of $A=0$, and a few cross-checks. It is a raw asymmetry, not corrected
for production and detection asymmetries.

| § | Content | Report section |
|---|---|---|
| 1 | Data, preselection, mass, counting asymmetry, cut flow | Data (add.: dataset and preselection) |
| 2 | Mass model and likelihood | Mass model and likelihood |
| 3 | Separate and simultaneous fits, goodness of fit, starting points, sideband subtraction | Frequentist analysis |
| 4 | Profile likelihood, Wilks test, polarities, binomial test | Frequentist analysis; Discussion |

The frequentist analysis continues in `02_Frequentist_validation`, `03_Frequentist_systematics` and `04_Dalitz_plane`.

**Scale.** A guided tour of the method, not the production run: replicas, toy experiments and chains are fewer than in
the report, so numbers can differ slightly from it; next to every result the notebook prints the value of the report
(**report:**). The notebook is self-contained and saves its figures in `figures/` and its numbers in `results/`.
Parts beyond the techniques of the course are tagged **[EXTRA]**.

## 0 · Setup

In [ ]:
import os, time, copy, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
from scipy.stats import norm, chi2
from iminuit import Minuit, cost

import plot_style
plot_style.use()
C = plot_style.COLORS
T0 = time.time()

# Data: put the three .root files of CERN Open Data record 4900 in ./data (or edit the path)
DATA_DIR = "data" if os.path.isdir("data") else "../../opendata-project/Data"
np.random.seed(1)

# figures and numerical results are saved next to the notebook
os.makedirs("figures", exist_ok=True); os.makedirs("results", exist_ok=True)
def save_fig(name):
    """Save the current figure in figures/ and show it."""
    plt.savefig(f"figures/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

M_K = 493.677            # charged-kaon mass [MeV]
LO, HI = 5100.0, 5600.0  # nominal fit window of the mass [MeV]

## 1 · Data and preselection

The record contains $B^\pm\to h^\pm h^+h^-$ candidates (2011 data, 7 TeV), one file per magnet polarity. Every
candidate has the momentum, the particle-identification (PID) likelihoods and the charge of the three tracks
$H_1,H_2,H_3$. We read only the branches we need.

In [ ]:
tracks = (1, 2, 3)                     # the three final-state hadrons H1, H2, H3
# branches to read for each track: momentum, PID likelihoods, muon flag, charge
branches = [f"H{i}_{v}" for i in tracks for v in ("PX", "PY", "PZ", "ProbK", "ProbPi", "isMuon", "Charge")]

frames = []
for pol in ("Down", "Up"):             # one file per magnet polarity
    with uproot.open(f"{DATA_DIR}/B2HHH_Magnet{pol}.root") as f:
        d = f["DecayTree"].arrays(branches, library="pd")      # read the tree into a pandas DataFrame
    d["magnet"] = pol.lower()          # remember the polarity of each candidate
    frames.append(d)
df = pd.concat(frames, ignore_index=True)
print("candidates in the record:", len(df))

**Preselection.** We want $B^\pm\to K^\pm K^+K^-$: all three tracks must look like kaons and not like pions or muons,

$$\texttt{ProbPi}<0.5\ \wedge\ \texttt{ProbK}>0.5\ \wedge\ \lnot\texttt{isMuon}\quad\text{for each of the three tracks.}$$

The cut flow below gives the effect of each requirement alone and cumulatively (additional result of the report).

In [ ]:
# boolean masks, True when the requirement is satisfied by ALL three tracks
not_muon = np.ones(len(df), bool)
pi_ok = np.ones(len(df), bool)
k_ok = np.ones(len(df), bool)
for i in tracks:
    not_muon &= ~df[f"H{i}_isMuon"].to_numpy().astype(bool)    # not identified as a muon
    pi_ok    &= df[f"H{i}_ProbPi"].to_numpy() < 0.5            # not pion-like
    k_ok     &= df[f"H{i}_ProbK"].to_numpy() > 0.5             # kaon-like

# cut flow: how many candidates pass each requirement alone, and applying them one after the other
print(f"{'requirement':28s} {'alone':>10s} {'cumulative':>11s}")
cum = np.ones(len(df), bool)
cutflow_rows = []
for name, mask in (("not muon (3 tracks)", not_muon), ("ProbPi < 0.5 (3 tracks)", pi_ok), ("ProbK > 0.5 (3 tracks)", k_ok)):
    cum &= mask
    print(f"{name:28s} {mask.sum():10d} {cum.sum():11d}")
    cutflow_rows.append((name, int(mask.sum()), int(cum.sum())))
print("report: 6311517 / 975724 / 28812 alone, 23895 cumulative")
pd.DataFrame(cutflow_rows, columns=["requirement", "alone", "cumulative"]).to_csv("results/01_cutflow.csv", index=False)

**Invariant mass.** We give the kaon mass to the three tracks, sum the four-momenta and take $m^2=E^2-|\vec p|^2$.
The two opposite-sign pairs $m(H_1H_2)$ and $m(H_1H_3)$ are kept for the Dalitz plane ($H_1$ is always the kaon with
charge opposite to the $B$).

We keep a slightly looser sample (`ProbK > 0.3`, `ProbPi < 0.7`) and store the *weakest* PID value of each
candidate: later the systematic study and the PID scan simply re-apply tighter cuts on these columns.

In [ ]:
loose = np.ones(len(df), bool)
for i in tracks:
    loose &= (df[f"H{i}_ProbK"] > 0.3) & (df[f"H{i}_ProbPi"] < 0.7) & ~df[f"H{i}_isMuon"].astype(bool)
d = df[loose].reset_index(drop=True)
del df

def invariant_mass(d, particles):
    """Invariant mass [MeV] of the listed tracks, kaon mass hypothesis."""
    E = sum(np.sqrt(d[f"H{i}_PX"]**2 + d[f"H{i}_PY"]**2 + d[f"H{i}_PZ"]**2 + M_K**2) for i in particles)
    px, py, pz = (sum(d[f"H{i}_{c}"] for i in particles) for c in ("PX", "PY", "PZ"))
    return np.sqrt(np.clip(E**2 - px**2 - py**2 - pz**2, 0, None))

d["m"]  = invariant_mass(d, (1, 2, 3))
d["m12"], d["m13"] = invariant_mass(d, (1, 2)), invariant_mass(d, (1, 3))
d["q"]  = d.H1_Charge + d.H2_Charge + d.H3_Charge              # charge of the B
d["pT"] = np.sqrt(sum(d[f"H{i}_PX"] for i in tracks)**2 + sum(d[f"H{i}_PY"] for i in tracks)**2)
d["p"]  = np.sqrt(d.pT**2 + sum(d[f"H{i}_PZ"] for i in tracks)**2)
d["min_probk"]  = d[[f"H{i}_ProbK" for i in tracks]].min(axis=1)
d["max_probpi"] = d[[f"H{i}_ProbPi" for i in tracks]].max(axis=1)
d = d[np.abs(d.q) == 1]
assert (d.H1_Charge == -d.q).all()          # H1 has the opposite charge of the B

nominal = d[(d.min_probk > 0.5) & (d.max_probpi < 0.5)]
sample = nominal[(nominal.m > LO) & (nominal.m < HI)].reset_index(drop=True)
print(f"preselected: {len(nominal)} (report 23895);  in the window [{LO:.0f},{HI:.0f}]: {len(sample)} (report 19731)")
m_pos = sample.m[sample.q > 0].to_numpy()       # B+
m_neg = sample.m[sample.q < 0].to_numpy()       # B-
n_pos, n_neg = len(m_pos), len(m_neg)

**A first number: counting.** Without any fit, the asymmetry of *all* candidates (signal and background together) is
$A_{\rm count}=(N^--N^+)/(N^-+N^+)$ with binomial uncertainty $\sqrt{(1-A^2)/N}$.

In [ ]:
def count_asym(q):
    """Counting asymmetry A = (N- - N+)/N of an array of B charges q = +-1, with its binomial uncertainty."""
    n = len(q)
    a = (np.sum(q < 0) - np.sum(q > 0)) / n
    return a, np.sqrt((1 - a**2) / n)

a_c, da_c = count_asym(sample.q.to_numpy())
print(f"N(B+) = {n_pos}, N(B-) = {n_neg};  A_count = {a_c:+.4f} +- {da_c:.4f}   (report: -0.0432 +- 0.0071)")

# mass spectrum: by charge (left) and by magnet polarity (right); log scale to see the background under the peak
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
bins = np.arange(LO, HI + 1, 5)
ax[0].hist(m_pos, bins, histtype="step", color=C["pos"], label="$B^+$")
ax[0].hist(m_neg, bins, histtype="step", color=C["neg"], label="$B^-$")
ax[0].set(yscale="log", xlabel="m(KKK) [MeV]", ylabel="candidates / 5 MeV"); ax[0].legend()
for pol in ("down", "up"):
    s = sample[sample.magnet == pol]
    ax[1].hist(s.m, bins, histtype="step", color=C[pol], label=f"magnet {pol} ({len(s)})")
ax[1].set(yscale="log", xlabel="m(KKK) [MeV]"); ax[1].legend()
plt.tight_layout(); save_fig("01_mass_spectrum")

The peak at $m\approx5284$ MeV over a falling background has the same shape for the two charges and the two
polarities: this allows a mass $\mu$ and a resolution $\sigma$ *common* to $B^+$ and $B^-$ in the model.

**Candidates sharing a track.** The record has no event identifier, so we check whether the same track appears in more
than one candidate (identical momentum): such candidates are not independent. They are few and all are kept.

In [ ]:
shares = np.zeros(len(sample), bool)
for i in tracks:
    shares |= sample.duplicated([f"H{i}_PX", f"H{i}_PY", f"H{i}_PZ"], keep=False).to_numpy()
print(f"candidates sharing a track with another candidate: {shares.sum()} ({100 * shares.mean():.1f}%)   (report: 42, 0.2%)")

## 2 · Mass model and likelihood

For each charge the mass spectrum is a mixture of signal and background on the window $[5100,5600]$ MeV,

$$f^\pm(m)=N_s^\pm\,S(m;\mu,\sigma)+N_b^\pm\,B^\pm(m),\qquad
N_s^\pm=\tfrac{N_s}{2}(1\mp A),$$

* **signal** $S$: Gaussian, truncated and normalised to the window; $\mu,\sigma$ are *shared* by $B^+$ and $B^-$;
* **background** $B^\pm$: sum of two exponentials, $f\,E(\lambda_1)+(1-f)\,E(\lambda_2)$, with a shape and a yield of
  its own for each charge (the steep exponential describes the rise of the background towards the lower edge);
* **$A$** is a parameter of the fit: it is the *parameter of interest* (POI); the other 11 are *nuisance parameters*.

The likelihood is *extended* (the yields are parameters, the number of candidates is Poisson):

$$-\ln\mathcal L=(N_s+N_b)-\sum_{\pm}\sum_{i\in\pm}\ln\!\big[N_s^\pm S(m_i)+N_b^\pm B^\pm(m_i)\big]+\text{const.}$$

We write the pdfs ourselves (a few lines each) and let `iminuit` build the extended likelihood with
`cost.ExtendedUnbinnedNLL`: the model is a function `model(m, *parameters)` that returns the expected total number
of events and the density of events at $m$.

In [ ]:
def gauss_pdf(m, mu, sigma, lo=LO, hi=HI):
    """Gaussian normalised on [lo, hi]."""
    return norm.pdf(m, mu, sigma) / (norm.cdf(hi, mu, sigma) - norm.cdf(lo, mu, sigma))

def exp_pdf(m, lam, lo=LO, hi=HI):
    """Decreasing exponential exp(-lam (m - lo)) normalised on [lo, hi]."""
    return lam * np.exp(-lam * (m - lo)) / (1 - np.exp(-lam * (hi - lo)))

def bkg_pdf(m, lam1, lam2, f, lo=LO, hi=HI):
    """Background: two exponentials, weight f on the steep one (lam1 > lam2)."""
    return f * exp_pdf(m, lam1, lo, hi) + (1 - f) * exp_pdf(m, lam2, lo, hi)

# one model per charge; the parameters with the same name (N_s, A, mu, sigma) are shared in the joint fit
def model_pos(m, N_s, A, mu, sigma, N_b_pos, lam1_pos, lam2_pos, f_pos):
    n_sig = N_s * (1 - A) / 2                       # B+ : (1 - A) / 2
    return n_sig + N_b_pos, n_sig * gauss_pdf(m, mu, sigma) + N_b_pos * bkg_pdf(m, lam1_pos, lam2_pos, f_pos)

def model_neg(m, N_s, A, mu, sigma, N_b_neg, lam1_neg, lam2_neg, f_neg):
    n_sig = N_s * (1 + A) / 2                       # B- : (1 + A) / 2
    return n_sig + N_b_neg, n_sig * gauss_pdf(m, mu, sigma) + N_b_neg * bkg_pdf(m, lam1_neg, lam2_neg, f_neg)

## 3 · The fit

### 3.1 Separate fits (a cross-check)

First each charge is fitted alone (seven free parameters: yields, $\mu$, $\sigma$, background shape) and the
asymmetry is computed from the two signal yields, $A=(N_s^--N_s^+)/(N_s^-+N_s^+)$, propagating the uncertainties.

In [ ]:
def model_single(m, N_s, mu, sigma, N_b, lam1, lam2, f):
    """Extended model of ONE charge: returns (expected number of events, density of events at m)."""
    return N_s + N_b, N_s * gauss_pdf(m, mu, sigma) + N_b * bkg_pdf(m, lam1, lam2, f)

def fit_single(m_data):
    """Extended unbinned fit of the masses of one charge (7 free parameters)."""
    nll = cost.ExtendedUnbinnedNLL(m_data, model_single)           # -2 ln L, built by iminuit
    fit = Minuit(nll, N_s=0.78 * len(m_data), mu=5284.0, sigma=18.0,   # starting values
                 N_b=0.22 * len(m_data), lam1=0.03, lam2=0.002, f=0.15)
    fit.limits["mu"] = (5260, 5310); fit.limits["sigma"] = (3, 40)       # physical ranges
    fit.limits["lam1"] = (1e-5, 0.1); fit.limits["lam2"] = (1e-5, 0.1); fit.limits["f"] = (0, 1)
    fit.limits["N_s"] = (0, 2 * len(m_data)); fit.limits["N_b"] = (0, 2 * len(m_data))
    fit.migrad()          # minimisation
    fit.hesse()           # uncertainties from the curvature (inverse Fisher information)
    return fit

fit_p, fit_n = fit_single(m_pos), fit_single(m_neg)
Np, dNp = fit_p.values["N_s"], fit_p.errors["N_s"]           # signal yield and error of B+
Nn, dNn = fit_n.values["N_s"], fit_n.errors["N_s"]           # and of B-
A_sep = (Nn - Np) / (Nn + Np)                                # asymmetry from the two yields
dA_sep = 2 * np.sqrt((Np * dNn)**2 + (Nn * dNp)**2) / (Nn + Np)**2    # error propagation, independent fits
print(f"valid: {fit_p.valid}, {fit_n.valid};  N_s(B+) = {Np:.0f} +- {dNp:.0f},  N_s(B-) = {Nn:.0f} +- {dNn:.0f}")
print(f"A_raw (separate fits) = {A_sep:+.4f} +- {dA_sep:.4f}     (report: -0.0442 +- 0.0087)")

**Reading the result.** The two charges fitted separately give practically the same asymmetry as the simultaneous fit below, so sharing $\mu$ and $\sigma$ between a particle and its antiparticle is adequate. The uncertainty of $A$ is dominated by the statistics of the signal yields.

### 3.2 Simultaneous fit: the nominal result

The two charges are fitted *together*: the total cost is the sum of the two extended likelihoods and $A$ is a
parameter of it, so the profile likelihood of $A$ is available directly. `Migrad` minimises, `Hesse` gives the
symmetric uncertainties from the curvature (inverse of the Fisher information) and `Minos` the asymmetric
intervals from the profile likelihood.

In [ ]:
def fit_joint(m_p, m_n, start, a_fixed=None):
    """Simultaneous fit of B+ and B-: the cost is the sum of the two extended likelihoods.
    With a_fixed the asymmetry is held at that value (profile likelihood at that point)."""
    nll = cost.ExtendedUnbinnedNLL(m_p, model_pos) + cost.ExtendedUnbinnedNLL(m_n, model_neg)
    f = Minuit(nll, **start)
    f.limits["A"] = (-1, 1); f.limits["mu"] = (5260, 5310); f.limits["sigma"] = (3, 40)
    f.limits["N_s"] = (0, 2 * (len(m_p) + len(m_n)))
    for s in ("pos", "neg"):
        f.limits[f"N_b_{s}"] = (0, 2 * (len(m_p) + len(m_n)))
        f.limits[f"lam1_{s}"] = (1e-5, 0.1); f.limits[f"lam2_{s}"] = (1e-5, 0.1); f.limits[f"f_{s}"] = (0, 1)
    if a_fixed is not None:
        f.values["A"] = a_fixed
        f.fixed["A"] = True
    f.migrad()
    return f

start = dict(N_s=0.78 * len(sample), A=-0.04, mu=5284.0, sigma=18.0,
             N_b_pos=0.22 * n_pos, lam1_pos=0.03, lam2_pos=0.002, f_pos=0.15,
             N_b_neg=0.22 * n_neg, lam1_neg=0.03, lam2_neg=0.002, f_neg=0.15)
fit = fit_joint(m_pos, m_neg, start)
best_vals_start = {k: fit.values[k] for k in start}      # best-fit values, used as starting point of later fits
fit.migrad(); fit.hesse()
fit.minos("A")                                      # 68 % interval (default cl = 1 sigma)
ci68 = (fit.values["A"] + fit.merrors["A"].lower, fit.values["A"] + fit.merrors["A"].upper)
fit_95 = copy.deepcopy(fit); fit_95.minos("A", cl=0.95)
ci95 = (fit_95.values["A"] + fit_95.merrors["A"].lower, fit_95.values["A"] + fit_95.merrors["A"].upper)

assert fit.valid and fit.values["lam1_pos"] > fit.values["lam2_pos"] and fit.values["lam1_neg"] > fit.values["lam2_neg"]
for k in ("N_s", "A", "mu", "sigma"):
    print(f"{k:8s} = {fit.values[k]:12.4f} +- {fit.errors[k]:.4f}")
print(f"N_b = {fit.values['N_b_pos'] + fit.values['N_b_neg']:.0f}")
print(f"\nA_raw = {fit.values['A']:+.4f} +- {fit.errors['A']:.4f} (Hesse)")
print(f"  68% Minos [{ci68[0]:+.4f}, {ci68[1]:+.4f}]    report: [-0.0526, -0.0356]")
print(f"  95% Minos [{ci95[0]:+.4f}, {ci95[1]:+.4f}]    report: [-0.0607, -0.0275]")
print("report: A = -0.0441 +- 0.0085, N_s = 15328 +- 133")

**Reading the result.** $A_{\rm raw}\approx-0.044$ with a Hesse uncertainty of $0.0085$: the asymmetry is about five standard deviations from zero. The Minos intervals are the ranges where the profile likelihood is within $1$ and $1.96^2$ units of its minimum; they are slightly asymmetric and the 95 % one does not contain zero.

In [ ]:
# table of the 12 parameters of the nominal fit, with their Hesse uncertainties, saved for later use
par_table = pd.DataFrame({"value": [fit.values[k] for k in fit.parameters], "hesse": [fit.errors[k] for k in fit.parameters]},
                         index=list(fit.parameters))
par_table.to_csv("results/01_fit_parameters.csv")
print(par_table.round(5).to_string())

**Different starting points.** A mixture model can have several minima. We refit from 10 random starting points
(yields, mass, resolution and background shape drawn at random): how many reach the same minimum?
*Report:* 29 of 30.

In [ ]:
n_same = 0
for k in range(30):
    # random starting point inside plausible ranges (yields, mass, resolution, background shape)
    st = dict(N_s=len(sample) * np.random.uniform(0.5, 0.95), A=np.random.uniform(-0.1, 0.1),
              mu=np.random.uniform(5270, 5300), sigma=np.random.uniform(8, 30),
              N_b_pos=n_pos * np.random.uniform(0.05, 0.5), lam1_pos=10**np.random.uniform(-2.5, -1.2),
              lam2_pos=10**np.random.uniform(-3.5, -2.6), f_pos=np.random.uniform(0.05, 0.5),
              N_b_neg=n_neg * np.random.uniform(0.05, 0.5), lam1_neg=10**np.random.uniform(-2.5, -1.2),
              lam2_neg=10**np.random.uniform(-3.5, -2.6), f_neg=np.random.uniform(0.05, 0.5))
    f_k = fit_joint(m_pos, m_neg, st)
    # same minimum = valid fit with the same value of -2 ln L (within 0.5)
    n_same += bool(f_k.valid and abs(f_k.fval - fit.fval) < 0.5)
print(f"{n_same} of 10 random starts reach the same minimum   (report: 29 of 30)")

**Reading the result.** All the random starts end in the same minimum: the likelihood has no competing local minima for this model, so the Migrad result is the global one.

### 3.3 Goodness of fit and picture

The $\chi^2=\sum_i(n_i-\nu_i)^2/\nu_i$ in bins of 10 MeV (both charges, $\nu_i$ from the fitted model) with
$\mathrm{ndof}=2\times50-12$. The pulls $(n_i-\nu_i)/\sqrt{\nu_i}$ show *where* the model fails. (The model is not
perfect: the signal tails are heavier than a Gaussian; $A_{\rm raw}$ does not depend on it, see §7.)

In [ ]:
def expected_counts(model, args, edges, n_sub=5):
    """Expected counts per bin: mean of the density at n_sub points in each bin, times the width."""
    width = edges[1] - edges[0]
    offs = (np.arange(n_sub) + 0.5) / n_sub                       # sub-points inside a bin
    pts = edges[:-1, None] + offs[None, :] * width                # shape (n_bins, n_sub)
    return model(pts, *args)[1].mean(axis=1) * width

edges = np.arange(LO, HI + 1e-9, 10.0)                            # bins of 10 MeV
centers = 0.5 * (edges[1:] + edges[:-1])
v = fit.values
# expected number of events in each bin for B+ and B-, from the fitted model
nu_pos = expected_counts(model_pos, [v[k] for k in ("N_s", "A", "mu", "sigma", "N_b_pos", "lam1_pos", "lam2_pos", "f_pos")], edges)
nu_neg = expected_counts(model_neg, [v[k] for k in ("N_s", "A", "mu", "sigma", "N_b_neg", "lam1_neg", "lam2_neg", "f_neg")], edges)
obs_pos, obs_neg = np.histogram(m_pos, edges)[0], np.histogram(m_neg, edges)[0]    # observed counts
chi2_val = np.sum((obs_pos - nu_pos)**2 / nu_pos) + np.sum((obs_neg - nu_neg)**2 / nu_neg)   # Pearson chi2
ndof = 2 * len(centers) - 12                                      # bins of both charges minus parameters
print(f"chi2/ndof = {chi2_val:.1f}/{ndof},  p = {chi2.sf(chi2_val, ndof):.1e}     (report: 160/88, p = 4.8e-6)")

# data with the fit (top) and pulls (bottom): the pulls show WHERE the model fails
pd.DataFrame({"m_center": centers, "obs_pos": obs_pos, "model_pos": nu_pos, "obs_neg": obs_neg, "model_neg": nu_neg}).to_csv(
    "results/01_mass_fit_bins.csv", index=False)    # data and model per bin
fig, ax = plt.subplots(2, 2, figsize=(12, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
for col, (obs, nu, lab, c) in enumerate(((obs_pos, nu_pos, "$B^+$", C["pos"]), (obs_neg, nu_neg, "$B^-$", C["neg"]))):
    ax[0, col].errorbar(centers, obs, np.sqrt(obs), fmt="o", ms=3, color="k", label="data")
    ax[0, col].plot(centers, nu, color=C["model"], label="fit")
    ax[0, col].set(yscale="log", ylabel="candidates / 10 MeV", title=lab); ax[0, col].legend()
    ax[1, col].bar(centers, (obs - nu) / np.sqrt(nu), width=9, color=c)
    ax[1, col].set(xlabel="m(KKK) [MeV]", ylabel="pull")
plt.tight_layout(); save_fig("01_joint_fit_pulls")

**Reading the result.** $\chi^2/{\rm ndof}$ is well above 1 and the $p$-value tiny: the model is **not** a perfect description of the spectrum (the pulls show structure on the flanks of the peak, where the signal tails are heavier than a Gaussian). What matters for the measurement is whether this affects $A$: the signal-model variants of notebook 03 say that it does not.

In [ ]:
# purity of the signal window |m - mu| < 2 sigma: signal / (signal + background) in the window, from the fit
mu_f, sg_f = fit.values["mu"], fit.values["sigma"]
x_w = np.linspace(mu_f - 2 * sg_f, mu_f + 2 * sg_f, 2001)
sig_in = fit.values["N_s"] * np.trapezoid(gauss_pdf(x_w, mu_f, sg_f), x_w)
bkg_in = sum(fit.values[f"N_b_{s}"] * np.trapezoid(bkg_pdf(x_w, fit.values[f"lam1_{s}"], fit.values[f"lam2_{s}"], fit.values[f"f_{s}"]), x_w)
             for s in ("pos", "neg"))
print(f"signal in the window: {sig_in:.0f}, background: {bkg_in:.0f}, purity {sig_in / (sig_in + bkg_in):.3f}   (report: about 96 %)")

## 4 · Profile likelihood and test of $A=0$

The **profile likelihood ratio** re-minimises the 11 nuisance parameters at each value of $A$,
$$q(A)=-2\ln\frac{\mathcal L(A,\hat{\hat\theta}(A))}{\mathcal L(\hat A,\hat\theta)} .$$
By Wilks' theorem $q(A_{\rm true})\sim\chi^2_1$, so the 68 % and 95 % intervals are the sets $q\le1$ and $q\le3.84$
(the Minos intervals above). The null hypothesis $H_0:A=0$ is tested with $q(0)$: $p=P(\chi^2_1>q(0))$ and the
significance is $Z=\sqrt{q(0)}$.

In [ ]:
def q_at(A0, best=fit):
    """Profile likelihood ratio at A = A0: fix A, re-minimise everything else."""
    f0 = copy.deepcopy(best)
    f0.values["A"] = A0
    f0.fixed["A"] = True
    f0.migrad()
    return f0.fval - best.fval            # iminuit cost functions return -2 ln L already

q0 = q_at(0.0)
print(f"q(0) = {q0:.1f},  p = {chi2.sf(q0, 1):.2e},  Z = sqrt(q) = {np.sqrt(q0):.2f} sigma     (report: q ~ 27, p = 2.05e-7, 5.20 sigma)")

a_grid, q_grid, _ = fit.mnprofile("A", size=30, bound=(-0.085, 0.0))
q_grid = q_grid - fit.fval            # fval = -2 ln L
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(a_grid, q_grid, "o-", color=C["freq"], ms=3)
ax.axhline(1, ls="--", color="gray"); ax.axhline(3.84, ls=":", color="gray")
ax.set(xlabel="$A_{\\rm raw}$", ylabel="$q(A)$", ylim=(0, 30))
plt.tight_layout(); save_fig("01_profile_likelihood")
pd.DataFrame({"A": a_grid, "q": q_grid}).to_csv("results/01_profile_likelihood.csv", index=False)

**Reading the result.** $q(0)\approx27$: the hypothesis $A=0$ is rejected at about $5\sigma$. The $p$-value is the probability of a result at least this extreme *if $A=0$*, not the probability that $A=0$. The significance refers to the **raw** asymmetry, not to CP violation.

**Exact binomial test** [EXTRA]. A test that uses no model of the mass at all: if there were no asymmetry, each
candidate would be $B^-$ with probability 1/2. It uses all candidates (signal and background), so it answers a
slightly different question.

**Why 6.1 sigma and not 5.2?** The count has a smaller uncertainty ($0.0071$ against $0.0085$) because it does not pay for
separating the background: it tests the charge asymmetry of *signal and background together*, without the mass. The fit tests
the asymmetry of the *signal alone*, with that of the background as a free parameter, and its $5.2\sigma$ is the number that
refers to the signal. The two are compatible (the asymmetry of the background is similar to that of the signal).

In [ ]:
from scipy.stats import binomtest
p_bin = binomtest(n_neg, n_pos + n_neg, 0.5).pvalue
print(f"exact binomial test on all candidates: p = {p_bin:.1e},  Z = {norm.isf(p_bin / 2):.1f} sigma   (report: 6.1 sigma)")

**Profile likelihood of the two magnet polarities.** The detector asymmetries change sign with the polarity. We fit the two
samples separately and *sum* the profile curves: since the two polarities are independent, $q_{\rm down}(A)+q_{\rm up}(A)$
is the profile likelihood of the combination (each polarity with its own nuisance parameters).

In [ ]:
prof = {}                                   # profile q(A) of each polarity
for pol in ("down", "up"):
    s_pol = sample[sample.magnet == pol]
    f_pol = fit_joint(s_pol.m[s_pol.q > 0].to_numpy(), s_pol.m[s_pol.q < 0].to_numpy(), best_vals_start)
    f_pol.hesse()
    a_pol, fv_pol, _ = f_pol.mnprofile("A", size=25, bound=(-0.12, 0.02))    # same grid of A for both polarities
    prof[pol] = fv_pol - f_pol.fval                                          # q(A) = -2 ln L - minimum
    print(f"magnet {pol:4s}: n = {len(s_pol):5d},  A = {f_pol.values['A']:+.4f} +- {f_pol.errors['A']:.4f}")
q_sum = prof["down"] + prof["up"]                   # independent samples: the log-likelihoods add up
inside = a_pol[q_sum <= q_sum.min() + 1]            # 68 % interval of the combination: q <= q_min + 1
print(f"sum of the profiles: A = {a_pol[np.argmin(q_sum)]:+.4f}, 68% interval [{inside.min():+.4f}, {inside.max():+.4f}]"
      f"      joint fit: {fit.values['A']:+.4f} +- {fit.errors['A']:.4f}")
print("report: down -0.0497, up -0.0378 (compatible at 0.7 sigma); the sum reproduces the joint fit")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(a_pol, prof["down"], color=C["down"], label="magnet down"); ax.plot(a_pol, prof["up"], color=C["up"], label="magnet up")
ax.plot(a_pol, q_sum, "k", label="sum"); ax.plot(a_grid, q_grid, color=C["freq"], ls="--", label="joint fit")
ax.axhline(1, color="gray", ls=":"); ax.set(xlabel="$A_{\\rm raw}$", ylabel="$q(A)$", ylim=(0, 8)); ax.legend()
plt.tight_layout(); save_fig("01_polarity_profiles")
pd.DataFrame({"A": a_pol, "q_down": prof["down"], "q_up": prof["up"], "q_sum": q_sum}).to_csv("results/01_polarity_profiles.csv", index=False)

### 3.4 Counting with sideband subtraction (a model-light cross-check) [EXTRA]

A different, simpler estimate: fit the *background alone* on the sidebands (window without
$[5200,5350]$ MeV), extrapolate it under the peak to get the expected background $b$ in the signal region, and
subtract it from the observed number of candidates. For each charge $S=N_{\rm obs}-b$, then
$A=(S^--S^+)/(S^-+S^+)$.

In [ ]:
SR = (5200.0, 5350.0)                                        # signal region
grid = np.linspace(LO, HI, 5001)

def sideband_background(m_side, sr=SR):
    """Fit of the two-exponential background on the sidebands (window without sr); returns the expected
    background in the signal region sr and the number of sideband events."""
    in_sb = (grid < sr[0]) | (grid > sr[1])
    in_sr = ~in_sb
    m_sb = m_side[(m_side < sr[0]) | (m_side > sr[1])]
    def pdf_sb(m, lam1, lam2, f):                           # background pdf renormalised on the sidebands
        norm_sb = np.trapezoid(bkg_pdf(grid, lam1, lam2, f)[in_sb], grid[in_sb])
        return bkg_pdf(m, lam1, lam2, f) / norm_sb
    f = Minuit(cost.UnbinnedNLL(m_sb, pdf_sb), lam1=0.03, lam2=0.002, f=0.15)
    f.limits["lam1"] = (1e-5, 0.1); f.limits["lam2"] = (1e-5, 0.1); f.limits["f"] = (0, 1)
    f.migrad()
    p = bkg_pdf(grid, *f.values)
    frac_sr = np.trapezoid(p[in_sr], grid[in_sr]) / np.trapezoid(p[in_sb], grid[in_sb])
    return len(m_sb) * frac_sr, len(m_sb)

S, dS = {}, {}
for lab, m_side in (("pos", m_pos), ("neg", m_neg)):
    b, n_sb = sideband_background(m_side)
    n_obs = np.sum((m_side > SR[0]) & (m_side < SR[1]))
    S[lab], dS[lab] = n_obs - b, np.sqrt(n_obs + b**2 / n_sb)
tot = S["pos"] + S["neg"]
a_sb = (S["neg"] - S["pos"]) / tot
da_sb = 2 * np.sqrt((S["pos"] * dS["neg"])**2 + (S["neg"] * dS["pos"])**2) / tot**2
print(f"A_raw (sideband subtraction) = {a_sb:+.4f} +- {da_sb:.4f}     (report: -0.0503 +- 0.0083)")

## Results saved

In [ ]:
summary = dict(
    counting=dict(A=float(a_c), err=float(da_c), n_pos=int(n_pos), n_neg=int(n_neg), candidates_sharing_a_track=int(shares.sum())),
    separate_fits=dict(A=float(A_sep), err=float(dA_sep)),
    joint_fit=dict(A=float(fit.values["A"]), hesse=float(fit.errors["A"]), minos68=[float(x) for x in ci68],
                   minos95=[float(x) for x in ci95], N_s=float(fit.values["N_s"]), mu=float(fit.values["mu"]),
                   sigma=float(fit.values["sigma"]), chi2=float(chi2_val), ndof=int(ndof)),
    test_of_A_zero=dict(q0=float(q0), p=float(chi2.sf(q0, 1)), Z=float(np.sqrt(q0))),
    binomial_test=dict(p=float(p_bin)), sideband_subtraction=dict(A=float(a_sb), err=float(da_sb)))
json.dump(summary, open("results/01_summary.json", "w"), indent=1)
print("saved results/01_summary.json; figures in figures/")

## Summary
* Counting gives a first number; the **simultaneous fit** separates signal from background: $A_{\rm raw}\approx-0.044\pm0.0085$
  (statistical), the same with separate fits and, roughly, with sideband subtraction.
* The **profile likelihood** and Wilks' theorem give the Minos intervals and reject $A=0$ at about $5\sigma$.
* The fit reaches the same minimum from different starting points; the model is not perfect ($\chi^2$ too large, signal
  tails heavier than a Gaussian), a point that the systematic study (notebook 03) shows to be harmless for $A_{\rm raw}$.
* The next notebook checks that the estimator is unbiased and the intervals cover.

In [ ]:
print(f"Running time of this notebook: {(time.time() - T0) / 60:.1f} min")